# Збір реальних метаданих: реклама в ШІ-системах та її вплив на користувачів

Ноутбук збирає метадані наукових публікацій з **OpenAlex**, **Crossref** та **arXiv**
(відкриті API, ключі не потрібні), зливає дублікати, кодує кожну роботу за схемою
`ai_ads_meta/taxonomy.py` і зберігає результат у файли для подальшого аналізу.

Порядок роботи: виконайте комірки згори донизу. Параметри збору — у розділі 2.
Потрібні `pandas` та доступ до інтернету.

## 1. Підключення пакета `ai_ads_meta`

Якщо ноутбук запущено поза репозиторієм (наприклад, у Colab), пакет буде склоновано з GitHub.

In [ ]:
import pathlib
import subprocess
import sys

REPO = "https://github.com/omega2417/bnt.git"
BRANCH = "claude/cool-sagan-b4s8zp"

def find_repo_root():
    here = pathlib.Path.cwd()
    for d in [here, *here.parents]:
        if (d / "ai_ads_meta" / "sources.py").exists():
            return d
    return None

root = find_repo_root()
if root is None:
    subprocess.run(["git", "clone", "--depth", "1", "-b", BRANCH, REPO, "bnt"], check=True)
    root = pathlib.Path("bnt").resolve()
if str(root) not in sys.path:
    sys.path.insert(0, str(root))

import pandas as pd

from ai_ads_meta import analyze, sources
from ai_ads_meta.model import dedupe, load_jsonl, save_jsonl
from ai_ads_meta.taxonomy import TAXONOMY, classify

print("Пакет:", root / "ai_ads_meta")

## 2. Параметри збору

- `QUERIES` — пошукові запити (англійською: так індексовано більшість робіт).
- `MAX_PER_QUERY` — скільки записів брати на один запит з кожного джерела.
  50 — швидка проба (~2 хв); для повного корпусу ставте 200–500.
- `MAILTO` — необов'язкова контактна адреса: OpenAlex і Crossref дають швидший
  доступ клієнтам, які її вказали. Адреса передається цим сервісам у заголовку запиту.

In [ ]:
import datetime

QUERIES = list(sources.DEFAULT_QUERIES)
# QUERIES += ["algorithmic anomie advertising", "AI companion advertising wellbeing"]

SOURCES = ["openalex", "crossref", "arxiv"]
FROM_YEAR = 2018
MAX_PER_QUERY = 50
MIN_RELEVANCE = 1      # 6 і більше відсікає слабко релевантні записи
MAILTO = ""

OUT_DIR = pathlib.Path("ai_ads_out") / datetime.date.today().isoformat()
OUT_DIR.mkdir(parents=True, exist_ok=True)

for q in QUERIES:
    print("-", q)
print("\nРезультати буде збережено в", OUT_DIR.resolve())

## 3. Збір даних

Запит, що завершився помилкою, пропускається (рядок `FAILED`), решта даних зберігається.

In [ ]:
raw = sources.collect(
    QUERIES, SOURCES, sources.make_fetch(MAILTO),
    from_year=FROM_YEAR, limit=MAX_PER_QUERY, log=sys.stdout,
)
print(f"\nЗібрано записів: {len(raw)}")
assert raw, "Нічого не зібрано: перевірте доступ до мережі та рядки FAILED вище"

## 4. Злиття дублікатів, кодування, збереження

In [ ]:
records = [classify(r) for r in dedupe(raw)]
save_jsonl(records, OUT_DIR / "records.jsonl")
summary = analyze.run(records, str(OUT_DIR), MIN_RELEVANCE)

print(f"Унікальних записів: {summary['collected']}")
print(f"Релевантних (згадано і ШІ, і рекламу): {summary['relevant']}")
print("\nФайли:")
for f in sorted(OUT_DIR.iterdir()):
    print(f"  {f.name:16} {f.stat().st_size / 1024:8.1f} КБ")

| Файл | Зміст |
|---|---|
| `records.jsonl` | усі унікальні записи з кодами |
| `relevant.csv` | релевантні записи, відсортовані за релевантністю |
| `features.csv` | бінарна матриця «публікація × категорія» |
| `summary.json`, `report.md` | зведена статистика |

## 5. Дані як таблиця `pandas`

`df` — усі записи, `rel` — релевантні. Колонки `ad_format`, `user_effect`, `method`,
`authors`, `countries`, `keywords` містять списки.

In [ ]:
df = pd.DataFrame([r.to_dict() for r in records])
rel = df[df["relevant"] & (df["relevance"] >= MIN_RELEVANCE)].sort_values(
    ["relevance", "cited_by"], ascending=False).reset_index(drop=True)

print(df.shape, "->", rel.shape)
rel[["title", "year", "venue", "cited_by", "relevance", "ad_format", "user_effect", "method", "framing"]].head(15)

## 6. Швидка перевірка зібраного

Коди присвоюються за ключовими словами, тож одна робота може мати кілька кодів, а частина — жодного.

In [ ]:
print("Заповненість полів серед релевантних записів, %")
filled = pd.Series({
    "анотація": (rel["abstract"].str.len() > 0).mean(),
    "DOI": (rel["doi"].str.len() > 0).mean(),
    "країни авторів": (rel["countries"].str.len() > 0).mean(),
    "кількість цитувань": rel["cited_by"].notna().mean(),
    "код впливу на користувачів": (rel["user_effect"].str.len() > 0).mean(),
}).mul(100).round(1)
display(filled.to_frame("%"))

display(rel["year"].value_counts().sort_index().to_frame("публікацій"))

In [ ]:
def code_counts(column):
    counts = rel[column].explode().value_counts()
    return pd.DataFrame({"n": counts, "%": (100 * counts / len(rel)).round(1)})

for column in TAXONOMY:
    display(code_counts(column).rename_axis(column))

In [ ]:
# Таблиця спряженості «формат реклами × вплив на користувачів»
pairs = rel[["ad_format", "user_effect"]].explode("ad_format").explode("user_effect").dropna().reset_index(drop=True)
pd.crosstab(pairs["ad_format"], pairs["user_effect"])

## 7. Подальший аналіз

Щоб повернутися до вже зібраних даних без повторного збору:

```python
records = load_jsonl("ai_ads_out/РРРР-ММ-ДД/records.jsonl")
features = pd.read_csv("ai_ads_out/РРРР-ММ-ДД/features.csv")
```

Перед використанням кодів у статті перевірте вручну вибірку записів з `relevant.csv`
і за потреби відкоригуйте правила в `ai_ads_meta/taxonomy.py`.